In [ ]:
import os
import subprocess
import sys
import tempfile

sys.path.insert(0, "/workspace/src")

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import torch

from datasets.cwb import get_zarr_dataset  # noqa: F401

from physicsnemo import Module
from physicsnemo.diffusion.generate import regression_step

plt.rcParams["animation.embed_limit"] = 100  # MB

# Data: stage the cut store to a local copy, then read locally.
#
# Reading the store directly over gs:// is NOT reliable here: the cut store is
# zarr v2 (make_cut_zarr.py pins zarr<3) but the container runs zarr v3, and over
# gcsfs zarr v3 fails to attach the v2 consolidated metadata, so member lookups
# raise `KeyError: 'cwb_valid'` / "Consolidated metadata ... not found". The
# training pipeline avoids this by staging to local SSD first; we do the same.
# scripts/stage_zarr.py downloads concurrently and strips any stray v3 zarr.json,
# producing a clean v2 store that zarr v3 reads via its `.zmetadata`.
GCS_DATA = "gs://norcorrdiff-us/taiwan_dataset/cwa_dataset/cwa_dataset_cut.zarr"
DATA_PATH = "/workspace/data/cwa_dataset_cut.zarr"  # persistent mount (~/ml-ds_data)

RESULTS_DIR = "gs://norcorrdiff-us/results/regression_cwb_20260623_130422"
CKPT_DIR = f"{RESULTS_DIR}/checkpoints_regression"

# One-time stage (~30 GB). Delete DATA_PATH to force a re-stage.
if not os.path.exists(DATA_PATH):
    print(f"Staging {GCS_DATA}\n     -> {DATA_PATH} (one-time, ~30 GB)...")
    subprocess.run(
        [sys.executable, "/workspace/scripts/stage_zarr.py",
         "--src", GCS_DATA, "--dst", DATA_PATH],
        check=True,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Validation split: the cwb dataset holds out year 2021 (train=False,
# all_times=False -> FilterTime(is_2021)). Channels / shape / add_grid / ds_factor
# MUST match the training config (cwb.yaml defaults + GCP overrides) so the model
# input/output line up with the checkpoint.
DATASET_KWARGS = dict(
    data_path=DATA_PATH,
    in_channels=[0, 1, 2, 3, 4, 9, 10, 11, 12, 17, 18, 19],
    out_channels=[0, 1, 2, 3],
    img_shape_x=128,
    img_shape_y=128,
    add_grid=False,
    ds_factor=1,
    train=False,       # validation split
    all_times=False,   # 2021 only
)

dataset = get_zarr_dataset(**DATASET_KWARGS)

assert len(dataset) > 0, (
    "Validation split is empty — the cut store may not span year 2021."
)

print(f"Dataset type   : {type(dataset).__name__}")
print(f"Dataset length : {len(dataset)}")
print(f"Image shape    : {dataset.image_shape()}")
print(f"Time range     : {dataset.time()[0]}  ->  {dataset.time()[-1]}")


In [ ]:
# Input (ERA5) and output (CWB) channels
input_channels = dataset.input_channels()
output_channels = dataset.output_channels()

print(f"Input channels  ({len(input_channels)}):")
for ch in input_channels:
    print(f"  {ch.name:<40} level={ch.level}")

print(f"\nOutput channels ({len(output_channels)}):")
for ch in output_channels:
    print(f"  {ch.name:<40} level={ch.level}")


In [ ]:
# Normalization stats for the selected channels
info = dataset.info()
era5_center, era5_scale = info["input_normalization"]
cwb_center, cwb_scale   = info["target_normalization"]

print(f"{'Input variable':<40} {'Center':>10} {'Scale':>10}")
print("-" * 62)
for ch, c, s in zip(input_channels, era5_center[list(dataset.in_channels)], era5_scale[list(dataset.in_channels)]):
    print(f"  {ch.name:<38} {c:10.4f} {s:10.4f}")

print(f"\n{'Output variable':<40} {'Center':>10} {'Scale':>10}")
print("-" * 62)
for ch, c, s in zip(output_channels, cwb_center[list(dataset.out_channels)], cwb_scale[list(dataset.out_channels)]):
    print(f"  {ch.name:<38} {c:10.4f} {s:10.4f}")


In [ ]:
# Sample shape and time range
y, x = dataset[0]
print(f"Input tensor  shape: {tuple(x.shape)}  (channels={x.shape[0]}, H={x.shape[1]}, W={x.shape[2]})")
print(f"Target tensor shape: {tuple(y.shape)}  (channels={y.shape[0]}, H={y.shape[1]}, W={y.shape[2]})")
print(f"\nTime range: {dataset.time()[0]}  →  {dataset.time()[-1]}")


In [ ]:
# Load the latest trained regression checkpoint from GCS.
# Checkpoints are named CorrDiffRegressionUNet.0.<nimg>.mdlus; pick the highest nimg.
# Module.from_checkpoint needs a local file, so download to a temp file first.
fs = fsspec.filesystem("gs")
ckpts = [p for p in fs.ls(CKPT_DIR) if p.endswith(".mdlus")]
assert ckpts, f"No .mdlus checkpoints found in {CKPT_DIR}"
latest = max(ckpts, key=lambda p: int(p.split(".")[-2]))  # nimg is 2nd-to-last field
print(f"Using checkpoint: gs://{latest}")

with tempfile.NamedTemporaryFile(suffix=".mdlus", delete=False) as tmp:
    tmp_path = tmp.name
try:
    with fs.open(latest, "rb") as remote_f, open(tmp_path, "wb") as local_f:
        local_f.write(remote_f.read())
    net = Module.from_checkpoint(tmp_path)
finally:
    os.unlink(tmp_path)

net = net.eval().to(device).to(memory_format=torch.channels_last)
if hasattr(net, "amp_mode"):
    net.amp_mode = False
print("Regression network loaded")


In [ ]:
# Inference helper: run the regression net on one validation sample and return
# truth/prediction in physical units. Mirrors src/generate_regression.py.
C_out = len(dataset.output_channels())
H, W = dataset.image_shape()


@torch.no_grad()
def predict(idx):
    """Return (truth, pred), each (C_out, H, W) in physical units."""
    target, input_ = dataset[idx]
    img_lr = input_[None].to(device).float().to(memory_format=torch.channels_last)
    out = regression_step(
        net=net,
        img_lr=img_lr,
        latents_shape=(1, C_out, H, W),
        lead_time_label=None,
    )
    truth = dataset.denormalize_output(target[None].numpy())[0]
    pred = dataset.denormalize_output(out.cpu().numpy())[0]
    return truth, pred


# Sanity check on the first validation sample
_truth, _pred = predict(0)
print(f"truth shape {_truth.shape}, pred shape {_pred.shape}")
print(f"finite values: truth={np.isfinite(_truth).all()}, pred={np.isfinite(_pred).all()}")

In [ ]:
from inference.visualization import (
    animate_variable,
    animate_wind_speed,
    plot_comparison,
    plot_variable,
)

In [ ]:
truth, pred = predict(0)
plot_comparison(truth, pred, dataset.output_channels(), dataset.time()[0], channel_idx=1)

In [ ]:
# Aggregate per-channel metrics over the validation set (physical units).
N_EVAL = min(len(dataset), 200)
if N_EVAL < len(dataset):
    print(f"Evaluating {N_EVAL} of {len(dataset)} validation samples")

out_names = [f"{c.name} {c.level}".strip() for c in dataset.output_channels()]

sse = np.zeros(C_out)   # sum of squared error per channel
sae = np.zeros(C_out)   # sum of absolute error
sbe = np.zeros(C_out)   # sum of error (for bias)
n_pix = 0

# Optional combined 10m wind-speed error (matches generate_regression.py treatment)
names_no_level = [c.name for c in dataset.output_channels()]
has_wind = "eastward_wind_10m" in names_no_level and "northward_wind_10m" in names_no_level
if has_wind:
    u_idx = names_no_level.index("eastward_wind_10m")
    v_idx = names_no_level.index("northward_wind_10m")
    ws_sse = 0.0

for i in range(N_EVAL):
    truth, pred = predict(i)
    err = pred - truth
    sse += (err ** 2).sum(axis=(1, 2))
    sae += np.abs(err).sum(axis=(1, 2))
    sbe += err.sum(axis=(1, 2))
    n_pix += truth.shape[1] * truth.shape[2]
    if has_wind:
        ws_t = np.sqrt(truth[u_idx] ** 2 + truth[v_idx] ** 2)
        ws_p = np.sqrt(pred[u_idx] ** 2 + pred[v_idx] ** 2)
        ws_sse += ((ws_p - ws_t) ** 2).sum()

rmse = np.sqrt(sse / n_pix)
mae = sae / n_pix
bias = sbe / n_pix

print(f"\nValidation metrics over {N_EVAL} samples (physical units):\n")
print(f"{'channel':<28} {'RMSE':>10} {'MAE':>10} {'bias':>10}")
print("-" * 60)
for name, r, m, b in zip(out_names, rmse, mae, bias):
    print(f"{name:<28} {r:10.4f} {m:10.4f} {b:10.4f}")

if has_wind:
    print(f"\n{'wind_speed_10m':<28} {np.sqrt(ws_sse / n_pix):10.4f}  (RMSE)")


In [ ]:
# Index into the validation set (must be < len(dataset))
sample_idx = 0

In [ ]:
# in_channels=[0,1,2,3,4,9,10,11,12,17,18,19] — original channel 17 is at subset index 9
plot_variable(dataset, "era5", channel_idx=9, sample_idx=sample_idx)


In [ ]:
plot_variable(dataset, "cwb", channel_idx=1, sample_idx=sample_idx)

In [ ]:
plot_variable(dataset, "pred", channel_idx=1, predict_fn=predict, sample_idx=sample_idx)

In [ ]:
animate_variable(dataset, "era5", channel_idx=9, start=24*7, n_steps=24*7)

In [ ]:
animate_variable(dataset, "cwb", channel_idx=1, start=24*7, n_steps=24*7)

In [ ]:
animate_variable(dataset, "pred", channel_idx=1, predict_fn=predict, start=24*7, n_steps=24*7)

In [ ]:
animate_wind_speed(dataset, "era5", start=24*7, n_steps=24*7)

In [ ]:
animate_wind_speed(dataset, "cwb", start=24*7, n_steps=24*7)

In [ ]:
animate_wind_speed(dataset, "pred", predict_fn=predict, start=24*7, n_steps=24*7)